# 杨氏模量计算器 (光杠杆 & 逐差法)

本 Notebook 用于通过光杠杆镜尺法测量金属丝在拉伸过程中的微小伸长量，利用**逐差法**计算金属丝的杨氏弹性模量 $E$，并进行完整的各分量不确定度传递与规范修约。

---
### 实验原理与数学公式

#### 1. 杨氏模量计算公式
根据胡克定律与光杠杆光学放大原理：
$$E = \frac{F/S}{\Delta L / L} = \frac{4 F L}{\pi d^2 \Delta L}$$
设单个砝码质量为 $m$，增重过程每加 4 个砝码（通过逐差法将 8 个加重、减重平均刻度值 $n_i$ 隔 4 项逐差）产生的刻度尺平均位移为 $\Delta n_{\mathrm{avg}}$，光杠杆后足长为 $b$，反射镜至标尺距离为 $D$，则伸长量 $\Delta L = \frac{b \Delta n_{\mathrm{avg}}}{2 D}$，代入得：
$$E = \frac{32 m g L D}{\pi d^2 b \Delta n_{\mathrm{avg}}}$$
* $L$：金属丝原长
* $D$：光杠杆镜面到刻度尺标尺距离
* $d$：金属丝直径（千分尺测量）
* $b$：光杠杆常数（刀口到后足支点的垂直距离）
* $m$：单组砝码质量；$g$：重力加速度
* $\Delta n_{\mathrm{avg}}$：4 个砝码引起的标尺平均位移

#### 2. 不确定度传递公式
* 相对不确定度平方和：
$$u_r(E) = \frac{u_E}{E} = \sqrt{ \left(\frac{u_L}{L}\right)^2 + \left(\frac{u_D}{D}\right)^2 + \left(2\frac{u_d}{d}\right)^2 + \left(\frac{u_b}{b}\right)^2 + \left(\frac{u_{\Delta n}}{\Delta n_{\mathrm{avg}}}\right)^2 }$$
* 各直接测量量的不确定度按 A 类与仪器 B 类合成：
$$u_X = \sqrt{u_{A,X}^2 + u_{B,X}^2}, \quad u_{A,X} = \frac{s_X}{\sqrt{n}}, \quad u_{B,X} = \frac{\Delta_{\mathrm{inst}}}{\sqrt{3}}$$

In [ ]:
import math
from decimal import Decimal
from python.utils import scientific_round, calculate_stats

print("杨氏模量模块加载完成。")

### 1. 实验参数、几何尺寸与标尺读数输入
> **提示**：可直接在下方单元格中修改各几何长度的多次测量列表以及加减砝码的刻度尺读数矩阵。

In [ ]:
# --- 常量设置 ---
M_WEIGHT = Decimal("0.350")  # 单个砝码质量 350g = 0.350kg
G = Decimal("9.80")          # 重力加速度 m/s^2
PI = Decimal(str(math.pi))

# --- 几何长度测量值 (单位: mm，各6次) ---
L_vals = [Decimal("905.0"), Decimal("905.5"), Decimal("904.5"), Decimal("905.0"), Decimal("905.2"), Decimal("904.8")]
D_vals = [Decimal("1502.0"), Decimal("1501.5"), Decimal("1502.5"), Decimal("1502.0"), Decimal("1501.8"), Decimal("1502.2")]
d_vals = [Decimal("0.702"), Decimal("0.704"), Decimal("0.701"), Decimal("0.703"), Decimal("0.702"), Decimal("0.703")]
b_vals = [Decimal("68.20"), Decimal("68.22"), Decimal("68.18"), Decimal("68.20"), Decimal("68.24"), Decimal("68.20")]

# --- 仪器误差限 Δ_inst (mm) ---
delta_L = Decimal("1.0")
delta_D = Decimal("1.0")
delta_d = Decimal("0.005")
delta_b = Decimal("0.02")
delta_n_inst = Decimal("0.5")

# --- 标尺读数 n (0~7 个砝码，单位: mm；每项为 [加重读数, 减重读数]) ---
n_readings = [
    [Decimal("15.2"), Decimal("15.4")],  # 0个
    [Decimal("22.1"), Decimal("22.3")],  # 1个
    [Decimal("29.0"), Decimal("29.1")],  # 2个
    [Decimal("35.8"), Decimal("36.0")],  # 3个
    [Decimal("42.7"), Decimal("42.9")],  # 4个
    [Decimal("49.5"), Decimal("49.7")],  # 5个
    [Decimal("56.3"), Decimal("56.5")],  # 6个
    [Decimal("63.1"), Decimal("63.2")],  # 7个
]

print("实验数据加载完成。")

### 2. 逐差法处理标尺读数

In [ ]:
# 计算各加重减重的平均读数 n_i
n_means = [(row[0] + row[1]) / Decimal("2") for row in n_readings]

# 逐差法计算 Δn (隔 4 项相减，相当于 4 个砝码的位移)
dn_list = []
print("--- 逐差过程 (Δn_j = n_{j+4} - n_j) ---")
for j in range(4):
    diff = n_means[j + 4] - n_means[j]
    dn_list.append(diff)
    print(f"Δn_{j} = n_{j+4} - n_{j} = {diff:.2f} mm")

dn_mean = sum(dn_list) / Decimal("4")
variance_dn = sum((x - dn_mean)**2 for x in dn_list) / Decimal("3")
s_dn = Decimal(str(math.sqrt(float(variance_dn))))
u_a_dn = s_dn / Decimal(str(math.sqrt(4)))
u_b_dn = delta_n_inst / Decimal(str(math.sqrt(3)))
dn_u = Decimal(str(math.sqrt(float(u_a_dn**2 + u_b_dn**2))))

print(f"\nΔn 平均值 : {dn_mean:.3f} ± {dn_u:.3f} mm")

### 3. 杨氏模量计算与不确定度评定

In [ ]:
# 几何参数的均值与不确定度
L_mean, L_u = calculate_stats(L_vals, delta_L)[:2]
D_mean, D_u = calculate_stats(D_vals, delta_D)[:2]
d_mean, d_u = calculate_stats(d_vals, delta_d)[:2]
b_mean, b_u = calculate_stats(b_vals, delta_b)[:2]

# 转换为国际单位制 (m)
L_m = L_mean / Decimal("1000")
D_m = D_mean / Decimal("1000")
d_m = d_mean / Decimal("1000")
b_m = b_mean / Decimal("1000")
dn_m = dn_mean / Decimal("1000")

# E = (32 * m * g * L * D) / (pi * d^2 * b * Δn)
numerator = Decimal("32") * M_WEIGHT * G * L_m * D_m
denominator = PI * (d_m**2) * b_m * dn_m
E_val = numerator / denominator

# 不确定度合成传递
rel_u_sq = (L_u/L_mean)**2 + (D_u/D_mean)**2 + (Decimal("2")*d_u/d_mean)**2 + (b_u/b_mean)**2 + (dn_u/dn_mean)**2
rel_u = Decimal(str(math.sqrt(float(rel_u_sq))))
E_u = E_val * rel_u

# 科学修约 (以 10^11 Pa 数量级表示)
E_11 = E_val / Decimal("1e11")
E_u_11 = E_u / Decimal("1e11")
E_final, u_final = scientific_round(E_11, E_u_11)

print("=" * 45)
print("             杨 氏 模 量 计 算 结 果          ")
print("=" * 45)
print(f"金属丝长度 L  : {L_mean:.2f} ± {L_u:.2f} mm")
print(f"镜面到尺距离 D: {D_mean:.2f} ± {D_u:.2f} mm")
print(f"金属丝直径 d  : {d_mean:.4f} ± {d_u:.4f} mm")
print(f"光杠杆臂长 b  : {b_mean:.3f} ± {b_u:.3f} mm")
print(f"标尺位移 Δn   : {dn_mean:.3f} ± {dn_u:.3f} mm (4个砝码对应位移)")
print("-" * 45)
print(f"杨氏模量 E (原始): {E_val:.4e} Pa")
print(f"相对不确定度 u_r : {rel_u * 100:.2f}%")
print(f"绝对不确定度 u_E : {E_u:.4e} Pa")
print("-" * 45)
print(f"★ 最终结果表示   : E = {E_final} ± {u_final} × 10^11 Pa")
print("=" * 45)